# Sesión complementaria: Red neuronal multicapa para el dataset Iris

**Curso:** TC2034 - Modelación del aprendizaje con inteligencia artificial

**Objetivos:**
- Aplicar los conceptos de redes neuronales multicapa a un problema de clasificación multiclase.
- Construir y entrenar una MLP con TensorFlow/Keras para clasificar las tres especies de flores Iris.
- Evaluar el rendimiento del modelo e interpretar sus predicciones usando SHAP.

**Dataset Iris:** Contiene 150 muestras de flores Iris con 4 características (longitud y ancho de sépalo y pétalo) y 3 clases (setosa, versicolor, virginica).

**Autor:** Adaptado del notebook original de Héctor Mejía-Díaz para fines educativos.

## 1. Carga y exploración de datos (ya pre-procesados)

El dataset Iris viene incluido en scikit-learn. Lo cargaremos y lo dividiremos en entrenamiento y prueba. Además, normalizaremos las características para un mejor entrenamiento de la red neuronal.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

# Cargar el dataset
iris = load_iris()
X = iris.data          # 150 muestras, 4 características
y = iris.target        # 0: setosa, 1: versicolor, 2: virginica
feature_names = iris.feature_names
target_names = iris.target_names

# Dividir en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Normalizar (escala estándar) para mejorar el entrenamiento
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Forma de X_train: {X_train.shape}")
print(f"Forma de y_train: {y_train.shape}")
print(f"Forma de X_test: {X_test.shape}")
print(f"Forma de y_test: {y_test.shape}")
print("Clases:", target_names)

## 2. Construcción de la red neuronal multicapa

Diseñaremos una MLP pequeña con:
- Capa de entrada: 4 neuronas (una por característica).
- Capa oculta: 8 neuronas con activación ReLU.
- Capa de salida: 3 neuronas con activación softmax (clasificación multiclase).

Usaremos el optimizador Adam y la función de pérdida `sparse_categorical_crossentropy` porque las etiquetas son enteros (no one-hot).

In [ ]:
model = keras.Sequential([
# Completar código

# Fin
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

## 3. Entrenamiento del modelo

Entrenaremos durante 100 épocas con un tamaño de lote de 8. Usaremos el 15% del entrenamiento como validación para monitorear overfitting.

In [ ]:
history = model.fit(X_train, y_train,
#Completar código

# Fin
                    verbose=1)

## 4. Evaluación del modelo

Evaluaremos el modelo en los datos de prueba y visualizaremos las curvas de aprendizaje.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Precisión en conjunto de prueba: {test_acc:.4f}")

# Graficar pérdida y accuracy durante el entrenamiento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], label='Entrenamiento')
ax1.plot(history.history['val_loss'], label='Validación')
ax1.set_xlabel('Épocas')
ax1.set_ylabel('Pérdida')
ax1.legend()
ax1.set_title('Curva de pérdida')

ax2.plot(history.history['accuracy'], label='Entrenamiento')
ax2.plot(history.history['val_accuracy'], label='Validación')
ax2.set_xlabel('Épocas')
ax2.set_ylabel('Precisión')
ax2.legend()
ax2.set_title('Curva de precisión')
plt.show()

## 5. Interpretación del modelo con SHAP

SHAP nos permite entender qué características (longitud de sépalo, ancho de pétalo, etc.) influyen más en la predicción de cada clase. Usaremos `GradientExplainer` (adecuado para redes neuronales entrenadas con TensorFlow).

In [ ]:
#!pip install shap -q

In [ ]:
import shap

# Seleccionar un subconjunto pequeño de prueba para agilizar
X_test_sample = X_test[:30]

# Explicador para redes neuronales
explainer = shap.GradientExplainer(model, X_train[:100])

shap_values = explainer.shap_values(X_test_sample)
# shap_values es una lista: cada elemento es un array de forma (n_samples, n_features) para una clase

# Visualización global: importancia media de las características por clase
shap.summary_plot(shap_values, X_test_sample, feature_names=feature_names, class_names=target_names)

## 6. Explicación de una predicción individual

Analicemos la primera muestra del conjunto de prueba: ¿qué características contribuyeron a que el modelo la clasificara como una especie determinada?

In [ ]:
idx = 0
prediccion = np.argmax(model.predict(X_test_sample[idx:idx+1]), axis=1)[0]
print(f"Muestra {idx}: características = {X_test_sample[idx]}")
print(f"Clase predicha: {target_names[prediccion]}")
print(f"Clase real: {target_names[y_test[idx]]}")

# Obtener valores SHAP para la clase predicha
shap_values_clase = shap_values[prediccion][idx]  # vector 1D

# Verificar longitud
if len(feature_names_correct) != len(shap_values_clase):
    print("¡Error de longitud! feature_names_correct tiene", len(feature_names_correct),
          "y shap_values_clase tiene", len(shap_values_clase))
    print("Usando nombres genéricos...")
    feature_names_correct = [f"Característica {i}" for i in range(len(shap_values_clase))]

# Usar shap.plots.bar con Explanation (sin expected_value)
# Para GradientExplainer, podemos poner base_values = 0 ya que solo nos interesa la magnitud relativa
explanation = shap.Explanation(values=shap_values_clase,
                               base_values=np.zeros_like(shap_values_clase),
                               data=X_test_sample[idx],
                               feature_names=feature_names_correct)
shap.plots.bar(explanation, show=False)
plt.title(f"Contribución de características para {target_names[prediccion]}")
plt.show()

## 7. Conclusión

Hemos construido una red neuronal multicapa que clasifica correctamente las flores Iris. A través de SHAP identificamos que la longitud del pétalo y el ancho del pétalo son las características más relevantes para la clasificación, lo cual es consistente con la literatura. Este flujo de trabajo puede extenderse a problemas más complejos como clasificación de imágenes o series temporales.